# SIRA Reproduction and Base64 Baseline on Colab L4

This notebook runs entirely on Google Colab. Your laptop GPU is not used.

Before running:

1. Open **Runtime > Change runtime type** and choose an **L4 GPU**.
2. Accept the licenses for both requested Meta Llama models on Hugging Face.
3. An `HF_TOKEN` Colab Secret is optional for the public smoke-test model.
4. Start with 5 or 10 samples. A 500-sample, seven-watermark reproduction is expensive and slow.


In [ ]:
# Check that Colab assigned the requested L4 GPU.
import subprocess

gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip()

print("GPU:", gpu_name)
if "L4" not in gpu_name:
    raise RuntimeError("This notebook requires an L4 GPU. Change the Colab runtime type and try again.")


In [ ]:
# Experiment settings. Keep the smoke test small before trying 500 samples.
REPO_URL = "https://github.com/hanifnoerr/Self-information-Rewrite-Attack.git"
BRANCH = "codex/browser-colab-l4"
REPO_DIR = "/content/Self-information-Rewrite-Attack"
OUTPUT_ROOT = "/content/sira_outputs"

ALGORITHM = "KGW"
SAMPLES = 10
# Public smoke-test model. This tests the pipeline but does not exactly reproduce the paper.
TINY_MODEL = "Qwen/Qwen2.5-3B-Instruct"
SMALL_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
RUN_SIRA_TINY = True
RUN_SIRA_SMALL = False
RUN_LLM_BASE64 = True
CALCULATE_PERPLEXITY = False

print(f"Algorithm: {ALGORITHM}")
print(f"Samples: {SAMPLES}")


In [ ]:
# Clone this adapted repository into /content.
from pathlib import Path
import shutil
import subprocess

repo_path = Path(REPO_DIR)
if repo_path.exists():
    shutil.rmtree(repo_path)

subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR],
    check=True,
)
print("Cloned repository to:", REPO_DIR)


In [ ]:
# Install the official requirements and the small additions used by this adaptation.
import subprocess

subprocess.run(
    ["pip", "install", "-r", f"{REPO_DIR}/requirements.txt"],
    check=True,
)
print("Requirements installed.")


In [ ]:
# Use HF_TOKEN when available. Public models work without one.
from google.colab import userdata
from huggingface_hub import hf_hub_download, login

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Using the HF_TOKEN Colab Secret.")
else:
    print("No HF_TOKEN found. Continuing with public models only.")

models_to_check = []
if RUN_SIRA_TINY or RUN_LLM_BASE64:
    models_to_check.append(TINY_MODEL)
if RUN_SIRA_SMALL:
    models_to_check.append(SMALL_MODEL)

for model_name in models_to_check:
    try:
        hf_hub_download(model_name, "config.json", token=hf_token)
        print("Model download access OK:", model_name)
    except Exception as error:
        print(f"Cannot download {model_name} with this HF_TOKEN.")
        print("Request access using the same Hugging Face account, then create a new read token.")
        raise


In [ ]:
# Save GPU, Python, and package details before the experiment.
import os
import subprocess

env = os.environ.copy()
env["PYTHONPATH"] = REPO_DIR
env["ALGORITHM"] = ALGORITHM
env["SAMPLES"] = str(SAMPLES)
env["TINY_MODEL"] = TINY_MODEL
env["SMALL_MODEL"] = SMALL_MODEL

subprocess.run(
    [
        "python", "scripts/write_environment.py",
        "--output_path", f"{OUTPUT_ROOT}/environment.json",
        "--tiny_model", TINY_MODEL,
        "--small_model", SMALL_MODEL,
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)


In [ ]:
# Run SIRA-Tiny with Llama 3.2 3B Instruct in bf16.
if RUN_SIRA_TINY:
    try:
        subprocess.run(
            ["bash", "scripts/run_sira_tiny_l4.sh"],
            cwd=REPO_DIR,
            env=env,
            check=True,
        )
    except subprocess.CalledProcessError:
        log_path = Path(OUTPUT_ROOT) / "logs" / "sira_tiny.log"
        if log_path.exists():
            print("\nLast 80 lines from", log_path)
            print("\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]))
        raise
else:
    print("Skipping SIRA-Tiny.")


In [ ]:
# Optionally run SIRA-Small with Llama 3 8B Instruct in 4-bit NF4.
if RUN_SIRA_SMALL:
    try:
        subprocess.run(
            ["bash", "scripts/run_sira_small_l4.sh"],
            cwd=REPO_DIR,
            env=env,
            check=True,
        )
    except subprocess.CalledProcessError:
        log_path = Path(OUTPUT_ROOT) / "logs" / "sira_small.log"
        if log_path.exists():
            print("\nLast 80 lines from", log_path)
            print("\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]))
        raise
else:
    print("Skipping SIRA-Small. Set RUN_SIRA_SMALL = True after the smoke test passes.")


In [ ]:
# Run raw Base64 encoding. Optionally ask the 3B model for Base64 paraphrases.
base64_command = [
    "python",
    "scripts/run_base64_baseline.py",
    "--input_path", f"{OUTPUT_ROOT}/watermarked/{ALGORITHM}_response.json",
    "--output_root", OUTPUT_ROOT,
    "--algorithm", ALGORITHM,
    "--max_samples", str(SAMPLES),
]

if RUN_LLM_BASE64:
    base64_command.extend([
        "--model_name", TINY_MODEL,
        "--dtype", "bf16",
        "--run_normal_paraphrase",
    ])

subprocess.run(base64_command, cwd=REPO_DIR, env=env, check=True)

if RUN_LLM_BASE64:
    subprocess.run(
        [
            "python", "scripts/decode_base64_outputs.py",
            "--input_path", f"{OUTPUT_ROOT}/base64_llm/base64_llm_raw.jsonl",
            "--output_path", f"{OUTPUT_ROOT}/base64_llm/base64_llm_decoded.jsonl",
        ],
        cwd=REPO_DIR,
        env=env,
        check=True,
    )


In [ ]:
# Evaluate available SIRA and Base64 outputs with the official detector.
evaluation_command = [
    "python", "scripts/evaluate_base64_baseline.py",
    "--generation_model", "facebook/opt-1.3b",
    "--algorithm", ALGORITHM,
    "--watermarked_input", f"{OUTPUT_ROOT}/watermarked/{ALGORITHM}_response.json",
    "--sira_tiny_input", f"{OUTPUT_ROOT}/sira_tiny/final/{ALGORITHM}_attack.json",
    "--sira_small_input", f"{OUTPUT_ROOT}/sira_small/final/{ALGORITHM}_attack.json",
    "--base64_raw_input", f"{OUTPUT_ROOT}/base64/base64_raw.jsonl",
    "--base64_decoded_input", f"{OUTPUT_ROOT}/base64/base64_decoded.jsonl",
    "--base64_llm_raw_input", f"{OUTPUT_ROOT}/base64_llm/base64_llm_raw.jsonl",
    "--base64_llm_decoded_input", f"{OUTPUT_ROOT}/base64_llm/base64_llm_decoded.jsonl",
    "--normal_paraphrase_input", f"{OUTPUT_ROOT}/normal_paraphrase/normal_paraphrase.jsonl",
    "--output_root", OUTPUT_ROOT,
    "--dtype", "bf16",
    "--max_samples", str(SAMPLES),
]

if CALCULATE_PERPLEXITY:
    evaluation_command.append("--calculate_ppl")

subprocess.run(evaluation_command, cwd=REPO_DIR, env=env, check=True)

subprocess.run(
    [
        "python", "scripts/compare_results.py",
        "--output_root", OUTPUT_ROOT,
        "--algorithm", ALGORITHM,
        "--samples", str(SAMPLES),
        "--tiny_model", TINY_MODEL,
        "--small_model", SMALL_MODEL,
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)


In [ ]:
# Show the final report and list every saved result file.
from pathlib import Path

report_path = Path(OUTPUT_ROOT) / "final_report.md"
print(report_path.read_text(encoding="utf-8"))

print("\nSaved output files:")
for path in sorted(Path(OUTPUT_ROOT).rglob("*")):
    if path.is_file():
        print(path)


In [ ]:
# Copy results to Google Drive so they survive after the Colab runtime stops.
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")
drive_output = Path("/content/drive/MyDrive/sira_outputs")

if drive_output.exists():
    shutil.rmtree(drive_output)

shutil.copytree(OUTPUT_ROOT, drive_output)
print("Copied results to:", drive_output)


## Stop the Runtime

After the Drive copy finishes, use **Runtime > Disconnect and delete runtime**. This releases the Colab GPU. Do not leave an L4 runtime idle.
